In [21]:
import math
import random
import pandas as pd
from collections import Counter
from pathlib import Path
import kagglehub

# Seed agar hasil konsisten
random.seed(42)


In [22]:
# fungsi dan prosedur 

def init_centroid(c1,c2):
    return [c1,c2]

# jarak 
def euclidean(a,b):
    return math.sqrt(
        sum((a[i]-b[i])**2 for i in range(len(a)))
    )

# clustering
def clustering(x,centroid):
    klaster = []
    for i in x:
        jarak = [euclidean(i,c) for c in centroid]
        klaster.append(jarak.index(min(jarak)))
    return klaster

# update centroid
def update_centroid(x,clusters,k):
    new_centroid = []
    for i in range(k):
        anggota = [
            x[j] for j in range(len(x)) 
            if clusters[j] == i 
        ]
        if not anggota:
            continue
        centroid = [
            sum(col)/len(col)
            for col in zip(*anggota)
        ]
        new_centroid.append(centroid)
    return new_centroid

# akurasi
def hitung_akurasi(y_asli, y_pred):
    benar = sum(1 for a, b in zip(y_asli, y_pred) if a == b)
    return benar / len(y_asli)

# RMSE
def hitung_rmse(y_asli, y_pred):
    se = sum((a - b) ** 2 for a, b in zip(y_asli, y_pred))
    return math.sqrt(se / len(y_asli))


In [23]:
# inisialisasi data

import kagglehub
from pathlib import Path

# Download latest version
path = kagglehub.dataset_download("dhairyajeetsingh/ecommerce-customer-behavior-dataset")

print("Path to dataset files:", path)

# cari file csv di folder hasil download
data = next(Path(path).glob("*.csv"), None)
if data is None:
    raise FileNotFoundError("file csv tidak ditemukan di hasil download kaggle")

# membaca data (sampling 1000 baris agar komputasi cepat dan rapi)
df = pd.read_csv(data, nrows=1000)

kolom_fitur = [
    "Average_Order_Value",
    "Total_Purchases",
    "Pages_Per_Session",
    "Membership_Years"
]

# mengambil data 
df = df.dropna(subset=kolom_fitur).copy()

# memastikan kolom ID tersedia untuk identifikasi data
if "ID" not in df.columns:
    df["ID"] = [f"P{i+1}" for i in range(len(df))]

x = df[kolom_fitur].astype(float).values.tolist()
id_pelanggan = df["ID"].tolist()

# inisialisasi centroid
k = 2
max_iter = 10

# menggunakan data pelanggan pertama dan terakhir
c1 = x[0].copy()
c2 = x[-1].copy()

centroid = init_centroid(c1, c2)

print("Jumlah data:", len(x))
print("Fitur:", kolom_fitur)
print(x[:3])
print("Centroid awal:")
print("C1:", c1)
print("C2:", c2)


Path to dataset files: C:\Users\haida\.cache\kagglehub\datasets\dhairyajeetsingh\ecommerce-customer-behavior-dataset\versions\1
Jumlah data: 935
Fitur: ['Average_Order_Value', 'Total_Purchases', 'Pages_Per_Session', 'Membership_Years']
[[94.72, 9.0, 6.0, 2.9], [82.45, 19.5, 10.3, 1.6], [165.52, 9.1, 1.6, 2.9]]
Centroid awal:
C1: [94.72, 9.0, 6.0, 2.9]
C2: [159.91, 9.1, 6.8, 4.6]


In [24]:
# main program

for iterasi in range(max_iter):
    # menyimpan centroid lama
    centroid_lama = [c.copy() for c in centroid]
    # menentukan klaster
    klaster = clustering(x,centroid)
    # menghitung centroid baru
    centroid = update_centroid(x,klaster,k)
    print(f"Iterasi ke-{iterasi+1}")
    for i, c in enumerate(centroid):
        print(f"Centroid {i+1}: {c}")
    # berhenti jika centroid tidak berubah
    if  centroid == centroid_lama:
        print("Centroid konvergen.")
        break


Iterasi ke-1
Centroid 1: [92.57331081081088, 12.932432432432444, 8.517905405405408, 3.030950685691646]
Centroid 2: [183.4434717352831, 13.308163265306122, 8.919825072886303, 2.9521097932415503]
Iterasi ke-2
Centroid 1: [97.4604613095238, 12.905803571428583, 8.509077380952377, 3.0336648897759737]
Centroid 2: [198.59726541901952, 13.490494296577944, 9.064638783269963, 2.921192620083087]
Iterasi ke-3
Centroid 1: [100.6840387275242, 12.977869986168747, 8.531535269709538, 2.991558042892543]
Centroid 2: [211.93373021321764, 13.385377358490567, 9.121698113207549, 3.0377358490566015]
Iterasi ke-4
Centroid 1: [102.60374167776293, 13.030359520639154, 8.554194407456716, 2.9994626697886924]
Centroid 2: [221.0277217674029, 13.233152173913046, 9.119021739130433, 3.0124999999999993]
Iterasi ke-5
Centroid 1: [104.77623559539043, 12.985275288092197, 8.569782330345706, 2.991672810513839]
Centroid 2: [233.07968055326057, 13.501298701298706, 9.150000000000002, 3.054545454545455]
Iterasi ke-6
Centroid 1: [

In [25]:
# Menentukan klaster berdasarkan centroid akhir
klaster = clustering(x, centroid)

df["Klaster"] = [
f"C{nomor + 1}" for nomor in klaster
]
df["Label"] = klaster

# Menyimpan seluruh data hasil klaster
nama_file = "dt_klaster_pelanggan.xlsx"
df.to_excel(nama_file, index=False)

print(f"\nHasil klaster berhasil disimpan ke file: {nama_file}")

print("\nCentroid akhir:")
for i, c in enumerate(centroid):
    print(f"C{i +1}: {c}")

print("\nHasil pengelompokan pelanggan (20 data pertama):")
print(df[["ID"] + kolom_fitur + ["Klaster"]].head(20).to_string(index=False))

print("\nJumlah anggota setiap klaster:")
print(df["Klaster"].value_counts().sort_index())



Hasil klaster berhasil disimpan ke file: dt_klaster_pelanggan.xlsx

Centroid akhir:
C1: [114.03691788526433, 13.07019122609674, 8.663217097862761, 3.0033706018124944]
C2: [355.34110446091546, 13.071739130434782, 8.706521739130435, 2.97608695652174]

Hasil pengelompokan pelanggan (20 data pertama):
 ID  Average_Order_Value  Total_Purchases  Pages_Per_Session  Membership_Years Klaster
 P1                94.72              9.0                6.0               2.9      C1
 P2                82.45             19.5               10.3               1.6      C1
 P3               165.52              9.1                1.6               2.9      C1
 P4               147.33             15.0               14.8               2.6      C1
 P5               190.97             12.0                6.9               3.5      C1
 P6               169.58             20.0               13.9               1.9      C1
 P7               161.98             13.0                8.3               1.6      C1
 P8 

In [26]:
# soal 1: langkah pengujian: split data latih/uji 80/20, hitung akurasi dan RMSE

# program K-NN tanpa library ML
def knn_predict_point(x_train, y_train, test_point, k_neighbors=3):
    jarak_list = [(euclidean(test_point, xt), yt) for xt, yt in zip(x_train, y_train)]
    jarak_list.sort(key=lambda item: item[0])
    tetangga_terdekat = [label for _, label in jarak_list[:k_neighbors]]
    return Counter(tetangga_terdekat).most_common(1)[0][0]

def knn_predict_all(x_train, y_train, x_test, k_neighbors=3):
    return [knn_predict_point(x_train, y_train, pt, k_neighbors) for pt in x_test]

# split data 80% data latih dan 20% data uji
idx = list(range(len(x)))
random.shuffle(idx)
batas = int(0.8 * len(x))
train_idx, test_idx = idx[:batas], idx[batas:]

x_latih = [x[i] for i in train_idx]
y_latih = [df["Label"].iloc[i] for i in train_idx]
x_uji = [x[i] for i in test_idx]
y_uji = [df["Label"].iloc[i] for i in test_idx]

# jalankan prediksi K-NN (k=3)
y_pred_knn = knn_predict_all(x_latih, y_latih, x_uji, k_neighbors=3)

acc_knn = hitung_akurasi(y_uji, y_pred_knn)
err_knn = hitung_rmse(y_uji, y_pred_knn)

print("Data latih:", len(x_latih), "(80%) | Data uji:", len(x_uji), "(20%)")
print("Akurasi data uji K-NN: %.4f (%.2f%%)" % (acc_knn, acc_knn * 100))
print("RMSE data uji K-NN   : %.4f" % err_knn)


Data latih: 748 (80%) | Data uji: 187 (20%)
Akurasi data uji K-NN: 1.0000 (100.00%)
RMSE data uji K-NN   : 0.0000


In [27]:
# soal 3: eksperimen perhitungan manual dengan 20 data pertama
x20 = x[:20]
id20 = id_pelanggan[:20]

# menggunakan data pertama dan terakhir dari 20 data sebagai centroid awal
c1_20 = x20[0].copy()
c2_20 = x20[-1].copy()
centroid20 = init_centroid(c1_20, c2_20)

print("== K-Means pada 20 Data Pertama ==")
for it in range(10):
    c_lama20 = [c.copy() for c in centroid20]
    klas20 = clustering(x20, centroid20)
    centroid20 = update_centroid(x20, klas20, 2)
    print(f"Iterasi ke-{it + 1}:")
    for i, c in enumerate(centroid20):
        print(f"  Centroid {i + 1}: {[round(v, 4) for v in c]}")
    if centroid20 == c_lama20:
        print("Centroid konvergen.\n")
        break

df20 = df.iloc[:20].copy()
df20["Klaster"] = [f"C{n + 1}" for n in klas20]
print("Hasil klaster 20 data:")
print(df20[["ID"] + kolom_fitur + ["Klaster"]].to_string(index=False))
print("\nJumlah anggota klaster 20 data:")
print(df20["Klaster"].value_counts().sort_index())


== K-Means pada 20 Data Pertama ==
Iterasi ke-1:
  Centroid 1: [95.8717, 10.65, 7.675, 2.05]
  Centroid 2: [148.8613, 14.7625, 9.85, 3.0625]
Iterasi ke-2:
  Centroid 1: [96.8446, 11.6769, 8.3154, 1.9846]
  Centroid 2: [154.6243, 13.4429, 8.9714, 3.3286]
Iterasi ke-3:
  Centroid 1: [100.398, 11.7867, 8.36, 2.44]
  Centroid 2: [167.076, 13.82, 9.1, 2.5]
Iterasi ke-4:
  Centroid 1: [100.398, 11.7867, 8.36, 2.44]
  Centroid 2: [167.076, 13.82, 9.1, 2.5]
Centroid konvergen.

Hasil klaster 20 data:
 ID  Average_Order_Value  Total_Purchases  Pages_Per_Session  Membership_Years Klaster
 P1                94.72              9.0                6.0               2.9      C1
 P2                82.45             19.5               10.3               1.6      C1
 P3               165.52              9.1                1.6               2.9      C2
 P4               147.33             15.0               14.8               2.6      C2
 P5               190.97             12.0                6.9       

In [28]:
# soal 4 & 5: perbandingan data baru: dihitung jarak ke centroid vs metode K-NN

def centroid_predict_point(centroids, test_point):
    jarak = [euclidean(test_point, c) for c in centroids]
    return jarak.index(min(jarak))

def centroid_predict_all(centroids, x_test):
    return [centroid_predict_point(centroids, pt) for pt in x_test]

# 1. Perbandingan pada data baru spesifik (Soal 4)
data_baru = [220.0, 15.0, 10.0, 3.0]
jarak_list = [(euclidean(data_baru, xt), yt) for xt, yt in zip(x_latih, y_latih)]
jarak_list.sort(key=lambda item: item[0])
top3_knn = jarak_list[:3]
vote_knn = Counter([yt for _, yt in top3_knn]).most_common(1)[0][0]

d_c1 = euclidean(data_baru, centroid[0])
d_c2 = euclidean(data_baru, centroid[1])
pred_cen_baru = 0 if d_c1 < d_c2 else 1

print("== 1. Pengujian pada Data Baru Spesifik ==")
print(f"Data Baru: {data_baru}")
print("3 Tetangga terdekat di data latih:")
for d, l in top3_knn:
    print(f"  - Jarak: {d:.4f} -> Klaster C{l + 1}")
print(f"Hasil prediksi K-NN       : C{vote_knn + 1}")
print(f"Jarak ke Centroid 1: {d_c1:.4f} | Jarak ke Centroid 2: {d_c2:.4f}")
print(f"Hasil prediksi Jarak Centroid: C{pred_cen_baru + 1}\n")

# 2. Evaluasi pada data uji (Soal 5)
y_pred_cen = centroid_predict_all(centroid, x_uji)

acc_cen = hitung_akurasi(y_uji, y_pred_cen)
err_cen = hitung_rmse(y_uji, y_pred_cen)

print(f"== 2. Evaluasi pada {len(x_uji)} Data Uji ==")
print("Metode K-NN (k=3)     : Akurasi=%.4f (%.2f%%)  RMSE=%.4f" % (acc_knn, acc_knn * 100, err_knn))
print("Metode Jarak Centroid : Akurasi=%.4f (%.2f%%)  RMSE=%.4f" % (acc_cen, acc_cen * 100, err_cen))
print()

# Cek data yang berbeda/salah prediksi
salah_knn = [i + 1 for i in range(len(y_uji)) if y_uji[i] != y_pred_knn[i]]
salah_cen = [i + 1 for i in range(len(y_uji)) if y_uji[i] != y_pred_cen[i]]
print("Data salah K-NN     :", salah_knn)
print("Data salah Centroid :", salah_cen)


== 1. Pengujian pada Data Baru Spesifik ==
Data Baru: [220.0, 15.0, 10.0, 3.0]
3 Tetangga terdekat di data latih:
  - Jarak: 3.1311 -> Klaster C1
  - Jarak: 5.5445 -> Klaster C1
  - Jarak: 7.2045 -> Klaster C1
Hasil prediksi K-NN       : C1
Jarak ke Centroid 1: 105.9891 | Jarak ke Centroid 2: 135.3610
Hasil prediksi Jarak Centroid: C1

== 2. Evaluasi pada 187 Data Uji ==
Metode K-NN (k=3)     : Akurasi=1.0000 (100.00%)  RMSE=0.0000
Metode Jarak Centroid : Akurasi=1.0000 (100.00%)  RMSE=0.0000

Data salah K-NN     : []
Data salah Centroid : []
